In [1]:
!pip install -q transformers datasets evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; plat

In [2]:
from datasets import load_dataset

# ✅ Load the final cleaned and merged 18-class dataset
dataset = load_dataset("Kanishkagarwal6101/Legal_Analyzer_Final")

label_list = sorted(set(dataset["train"]["label"]))
num_labels = len(label_list)

print(f"✅ Labels: {label_list}")
print(f"✅ Number of classes: {num_labels}")


README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/7.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38831 [00:00<?, ? examples/s]

✅ Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
✅ Number of classes: 18


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_ckpt = "allenai/longformer-base-4096"  # Longformer base model
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# ✅ Model setup for classification
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels)


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=4096)
    tokens["labels"] = examples["label"]
    return tokens

tokenized_ds = dataset.map(tokenize_function, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])

# ✅ Train/Validation Split
if "validation" not in tokenized_ds:
    split = tokenized_ds["train"].train_test_split(test_size=0.1, seed=42)
    tokenized_ds["train"] = split["train"]
    tokenized_ds["validation"] = split["test"]

print(f"✅ Train Size: {len(tokenized_ds['train'])} | Validation Size: {len(tokenized_ds['validation'])}")


Map:   0%|          | 0/38831 [00:00<?, ? examples/s]

✅ Train Size: 34947 | Validation Size: 3884


In [5]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./longformer-final",
    eval_strategy="steps",        # ✅ This line is critical
    save_strategy="steps",              # ✅ Matches eval strategy
    eval_steps=500,                     # Evaluate every 500 steps
    save_steps=500,                     # Save every 500 steps
    per_device_train_batch_size=1,      # ⚠️ Longformer inputs are huge
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      # Effective batch size = 8
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                          # ✅ Mixed precision = faster on A100
    save_total_limit=2
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
# ✅ Save model
trainer.save_model("./longformer-final")
tokenizer.save_pretrained("./longformer-final")
print("✅ Model saved at ./longformer-final")


<ipython-input-10-ef7b703921c2>:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kanishk6101 (kanishk6101-purdue-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Initializing global attention on CLS token...


Step,Training Loss,Validation Loss,Accuracy,F1
500,1.225600,1.264333,0.545314,0.536841
1000,1.269800,1.184556,0.588311,0.571262
1500,1.067200,1.179356,0.607364,0.583877
2000,1.119800,1.085289,0.621524,0.610244
2500,1.088800,1.036199,0.641349,0.647643
3000,1.028500,1.021551,0.638260,0.641905
3500,1.109400,0.968857,0.655767,0.654194
4000,1.066900,0.985586,0.660917,0.670258
4500,0.967500,0.929795,0.683574,0.691709
5000,0.691400,0.986399,0.679197,0.678869


✅ Model saved at ./longformer-final


In [11]:
trainer.save_model("./longformer-final")
tokenizer.save_pretrained("./longformer-final")
print("✅ Model saved at ./longformer-final")

✅ Model saved at ./longformer-final


In [12]:
import shutil

# Zip the saved model directory
shutil.make_archive("longformer-final", "zip", "./longformer-final")
print("✅ Zipped model as longformer-final.zip")
from google.colab import files

# Download it to your local machine
files.download("longformer-final.zip")


✅ Zipped model as longformer-final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!pip install PyPDF2
import torch
import nltk
import numpy as np
import pandas as pd
import re
from PyPDF2 import PdfReader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ✅ Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nltk.download("punkt")

# ✅ Load your fine-tuned Longformer model
model_path = "./longformer-final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

# ✅ Label map for 18-class setting
label2id = {
    "Business": 0, "Confidentiality": 1, "Consumers": 2, "Declarations": 3, "Economy": 4,
    "Education": 5, "Employment": 6, "Environment": 7, "External Relations": 8, "Fairness": 9,
    "Health": 10, "IP & Rights": 11, "Indemnification": 12, "Legal Governance": 13,
    "Miscellaneous": 14, "Payment": 15, "Social": 16, "Termination": 17
}
id2label = {v: k for k, v in label2id.items()}

# ✅ Load and clause-chunk the PDF
reader = PdfReader("/content/independent_contractor_agreement (1).pdf")
text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
clause_chunks = re.split(r"\n?\s*\d+\.\s+", text)
clause_chunks = [chunk.strip() for chunk in clause_chunks if chunk.strip()]
print(f"✅ Total clause-based chunks: {len(clause_chunks)}")

# ✅ Predict
predicted_ids = []
for clause in clause_chunks:
    inputs = tokenizer(clause, return_tensors="pt", truncation=True, padding=True, max_length=4096).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        predicted_id = torch.argmax(logits, dim=1).item()
    predicted_ids.append(predicted_id)

predicted_labels = [id2label[i] for i in predicted_ids]

# ✅ True labels for the known chunks (adjusted to your contract)
expected_labels = [
    "Business",            # Intro
    "Employment",          # Scope
    "Termination",         # Term
    "Payment",             # Compensation
    "Confidentiality",     # NDA
    "IP & Rights",         # Ownership
    "Indemnification",     # Liability
    "Legal Governance",    # Governing law
    "Miscellaneous"        # Final
]

# ✅ Align lengths
expected_labels = expected_labels[:len(clause_chunks)]
true_ids = [label2id[label] for label in expected_labels]
predicted_ids = predicted_ids[:len(expected_labels)]
predicted_labels = predicted_labels[:len(expected_labels)]
clause_chunks = clause_chunks[:len(expected_labels)]

# ✅ Metrics
acc = accuracy_score(true_ids, predicted_ids)
prec, rec, f1, _ = precision_recall_fscore_support(true_ids, predicted_ids, average="weighted", zero_division=0)

# ✅ Output
print("\n📊 Longformer Clause-Aware Evaluation:")
print(f"✅ Accuracy: {acc * 100:.2f}%")
print(f"✅ Precision: {prec * 100:.2f}%")
print(f"✅ Recall: {rec * 100:.2f}%")
print(f"✅ F1 Score: {f1 * 100:.2f}%")

# ✅ Save breakdown
df_eval = pd.DataFrame({
    "Clause": clause_chunks,
    "Predicted": predicted_labels,
    "Expected": expected_labels
})
df_eval.to_csv("longformer_clause_aware_eval.csv", index=False)
print("\n📁 Saved to longformer_clause_aware_eval.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


✅ Total clause-based chunks: 9

📊 Longformer Clause-Aware Evaluation:
✅ Accuracy: 77.78%
✅ Precision: 70.37%
✅ Recall: 77.78%
✅ F1 Score: 72.22%

📁 Saved to longformer_clause_aware_eval.csv
